In [30]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Optional, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
import operator
load_dotenv()


True

In [31]:
class TweetState(TypedDict):
    topic: str
    iteration_count: int
    max_iterations: int
    tweet: str
    response: Literal["approved", "needs_improvement"]
    feedback: str
    

    tweet_history: Annotated[list[str], Field(description="History of generated tweets for this topic"), operator.add]
    tweet_feedback_history: Annotated[list[str], Field(description="History of feedback for generated tweets for this topic"), operator.add]

In [32]:
class EvaluationSchema(BaseModel):
    response: Literal["approved", "needs_improvement"] = Field(description="The evaluator's decision on whether the generated tweet is approved or needs improvement.")
    feedback: str = Field(description="Detailed feedback from the evaluator on the generated tweet, including specific reasons for the decision and suggestions for improvement if applicable.")

In [33]:
gen_llm = ChatGroq(model="llama-3.3-70b-versatile")
eval_llm = ChatGroq(model="llama-3.3-70b-versatile")
optimizer_llm = ChatGroq(model="llama-3.3-70b-versatile")

structured_eval_model = eval_llm.with_structured_output(EvaluationSchema)

In [ ]:
def generate_tweet(state: TweetState) -> TweetState:
    prompt = [
        SystemMessage(content="You are a social media manager tasked with creating engaging tweets on various topics. Your goal is to generate a tweet based on the given topic and feedback history, ensuring that the content is relevant and appealing to the target audience."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nTweet History: {state['tweet_history']}\n\nFeedback History: {state['tweet_feedback_history']}\n\nPlease generate a tweet based on the topic and feedback history.")
    ]
    response = gen_llm.invoke(prompt)
    iteration_count = state['iteration_count'] + 1

    return {"tweet": response.content, "tweet_history": [response.content], "iteration_count": iteration_count}

In [35]:
def evaluate_tweet(state: TweetState) -> TweetState:
    prompt = [
        SystemMessage(content="You are a social media content evaluator responsible for assessing the quality of generated tweets based on a given topic and feedback history. Your task is to evaluate the provided tweet and determine whether it is approved or needs improvement, along with detailed feedback."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nGenerated Tweet: {state['tweet']}\n\nTweet History: {state['tweet_history']}\n\nFeedback History: {state['tweet_feedback_history']}\n\nPlease evaluate the generated tweet and provide your response in the specified format.")
    ]
    response = structured_eval_model.invoke(prompt)

    return {"response": response.response, "feedback": response.feedback, "tweet_feedback_history": [response.feedback]}

In [36]:
def check_approval(state: TweetState):
    if state['response'] == "approved" or state['iteration_count'] >= state['max_iterations']:
        return "approved"
    else:     
        return "needs_improvement"

In [37]:
def optimize_tweet(state: TweetState) -> TweetState:
    prompt = [
        SystemMessage(content="You are a skilled social media strategist tasked with optimizing tweets based on evaluator feedback. Your goal is to take the generated tweet and the detailed feedback provided by the evaluator to create an improved version of the tweet that addresses the concerns raised and enhances its appeal to the target audience."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nGenerated Tweet: {state['tweet']}\n\nEvaluator Feedback: {state['feedback']}\n\nPlease optimize the generated tweet based on the evaluator's feedback and provide an improved version.")
    ]
    response = optimizer_llm.invoke(prompt)
    iteration_count = state['iteration_count'] + 1

    return {"tweet": response.content, "tweet_history": [response.content], "iteration_count": iteration_count}

In [38]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", check_approval, {
    "approved": END,
    "needs_improvement": "optimize_tweet"
})

graph.add_edge("optimize_tweet", "evaluate_tweet")




In [39]:
workflow = graph.compile()
initial_state = {
    "topic": "The benefits of using AI in daily life",
    "iteration_count": 0,
    "max_iterations": 5,
}
result = workflow.invoke(initial_state)
print(result)

{'topic': 'The benefits of using AI in daily life', 'iteration_count': 1, 'max_iterations': 5, 'tweet': 'Here\'s a tweet on the benefits of using AI in daily life:\n\n"Discover the power of AI in your daily routine! From virtual assistants to personalized recommendations, AI is making our lives easier, faster, and more efficient. What\'s your favorite way to use AI in your daily life? #AI #Innovation #Technology"', 'response': 'approved', 'feedback': 'The generated tweet is well-structured, informative, and engaging. It effectively highlights the benefits of using AI in daily life, and the use of hashtags #AI, #Innovation, and #Technology can help increase its reach. The tweet also encourages user interaction by asking a question, which can lead to a higher level of engagement. Overall, the tweet is of high quality and is approved.', 'tweet_history': ['Here\'s a tweet on the benefits of using AI in daily life:\n\n"Discover the power of AI in your daily routine! From virtual assistants 